# Chapter 9 - Lab 2: <font color='blue'>Return-Abuse Investigation via Adversarial Debate</font> (customer-care edition)

**<font color='purple'>Goal</font>**:
When the Lab 1 abuse screener flags a claim, you need a deeper look before treating a customer
as a suspect. This lab implements that investigation with an **adversarial debate**:

* an **Abuse Advocate** argues the claim is deliberate abuse,
* a **Customer Advocate** argues it is genuine, offering an innocent explanation for every red flag,
* a senior **Investigation Decision** agent weighs both arguments and produces a referral signal.

The case: **Vikram Singh** (C1004) says his new Rs 74,990 TV arrived with a cracked screen and
wants a refund to a different card. He has three "damaged in transit" claims in six months.

Forcing a model to argue *both* sides surfaces evidence quality and weak reasoning that a
one-shot "is this abuse?" classifier would miss. Each advocate must also admit *the weaknesses in
its own case*. In customer care this matters twice over: wrongly accusing an honest customer is
a brand and legal risk, while paying out every claim invites more abuse.

| Finance (insurance) edition | Customer-care edition |
|---|---|
| Fraud Advocate | **AbuseAdvocate** |
| Legitimacy Advocate (consumer protection) | **CustomerAdvocate** |
| Investigation Decision | **InvestigationDecision** (trust and safety) |
| `strong_fraud_indicators` ... `likely_legitimate` | `strong_abuse_indicators` ... `likely_legitimate` |

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex**: `FunctionAgent` with explicit handoffs.
* **OpenAI** `gpt-4o`.
* **Pydantic**: an optional `AbuseArgument` schema from `common.py`; the agents here return free text.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared models and helpers

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%209/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import AbuseArgument, SUSPICIOUS_CLAIM_SUMMARY

print('Common helpers loaded.')
print('Claim under investigation:\n' + SUSPICIOUS_CLAIM_SUMMARY)

## 4. Define the three debate agents

The two advocates have no tools: they reason from the claim summary. Their prompts force them to
(a) build the strongest case for their side and (b) admit their own weaknesses.

The **handoff chain** is sequential: `AbuseAdvocate -> CustomerAdvocate -> InvestigationDecision`.
Without this explicit chain, `AgentWorkflow` would never invoke the Customer Advocate: agents are
only reachable via a handoff or as the root.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)

abuse_advocate = FunctionAgent(
    name='AbuseAdvocate',
    description='Argue the case for return or warranty abuse.',
    system_prompt=(
        'You are a returns-abuse investigator at an electronics retailer. Review the claim and '
        'build the strongest possible case that it is deliberate abuse (false damage claim, '
        'refund fraud). Cite specific evidence. Also identify the weaknesses in your own '
        'argument: where is your case thin? Be thorough but honest. When done, hand off to '
        'CustomerAdvocate so the other side can respond.'
    ),
    llm=llm, tools=[], can_handoff_to=['CustomerAdvocate'],
)

customer_advocate = FunctionAgent(
    name='CustomerAdvocate',
    description='Argue the claim is genuine.',
    system_prompt=(
        'You are a consumer-protection advocate. For every red flag the AbuseAdvocate raised, '
        'provide an innocent alternative explanation. Build the strongest possible case that '
        'the claim is genuine, and acknowledge your own weaknesses. Your role is to make sure '
        'no honest customer is wrongly refused or treated as a suspect. When done, hand off '
        'to InvestigationDecision.'
    ),
    llm=llm, tools=[], can_handoff_to=['InvestigationDecision'],
)

investigation_decision = FunctionAgent(
    name='InvestigationDecision',
    description='Weigh the abuse and customer arguments.',
    system_prompt=(
        'You are the senior trust-and-safety investigator. You have arguments from an Abuse '
        'Advocate and a Customer Advocate. Evaluate (1) evidence quality, (2) reasoning '
        'strength, (3) acknowledged weaknesses on each side. Produce a referral signal: '
        'strong_abuse_indicators / likely_abuse / inconclusive / likely_legitimate. Include a '
        'confidence (0-1) and a recommended next action (for example replacement after '
        'inspection, refund only to the original payment method, or referral to the '
        'loss-prevention team). A person makes the final call.'
    ),
    llm=llm, tools=[], can_handoff_to=[],
)

## 5. Wire the debate as a workflow

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

workflow = AgentWorkflow(
    agents=[abuse_advocate, customer_advocate, investigation_decision],
    root_agent='AbuseAdvocate',
    initial_state={'claim_summary': SUSPICIOUS_CLAIM_SUMMARY},
)

## 6. Run the debate

The handoff chain drives the debate: AbuseAdvocate argues and hands off to CustomerAdvocate, who responds and hands off to InvestigationDecision, who concludes.

In [ ]:
user_msg = (
    'Investigate the following potentially abusive claim. Argue both sides '
    'before reaching a determination.\n\n'
    f'Claim summary:\n{SUSPICIOUS_CLAIM_SUMMARY}'
)

handler = workflow.run(user_msg=user_msg)

last = None
async for event in handler.stream_events():
    name = getattr(event, 'current_agent_name', None)
    if name and name != last:
        print(f'\n=== {name} ===')
        last = name
    if getattr(event, 'delta', None):
        print(event.delta, end='', flush=True)

final = await handler
print('\n\nFinal determination:\n' + str(final))

## 7. Structured arguments (optional, advanced)

In production you would want each advocate to return a **typed** `AbuseArgument` instead of free text, so the judge (and an auditor) can compare evidence lists side by side. The schema from `common.py`:

In [ ]:
import json

sample_abuse_argument = AbuseArgument(
    position='abuse',
    confidence=0.75,
    evidence=[
        'Courier proof-of-delivery photo shows an intact box, signed without remarks',
        "Three 'damaged in transit' claims in six months",
        'Refund requested to a different card than the one used to pay',
        'Chargeback threat within four days of delivery',
    ],
    reasoning='Pattern matches serial false-damage claims followed by refund diversion.',
    weaknesses=[
        'A point-impact crack can happen inside an intact box if the carton was dropped.',
        'The panel serial number matches the invoice, so this is not a swapped unit.',
    ],
)
print(json.dumps(sample_abuse_argument.model_dump(), indent=2))

## 8. Results

**What to notice about the adversarial-debate pattern:**

* **Better than a single-pass classifier.** Forcing both arguments surfaces evidence quality and
  weak reasoning that a one-shot prompt would miss.
* **Admitted weaknesses anchor the decision.** When both sides must name their own weak points,
  the judge has something concrete to weigh.
* **The output is a referral signal, not a verdict on the customer.** A person decides; the
  debate makes that person faster and better informed.
* **Cheap first, deep second.** Lab 1's AbuseScreener flags suspects with a fast rule score. Only
  when the score crosses the threshold does this (slower, costlier) debate run.

**Discussion questions for class**

1. What is the business cost of a false positive (honest customer treated as a fraudster) versus a false negative here?
2. Which recommended actions are safe to automate (for example "refund only to the original payment method") and which are not?
3. Should the same model play both advocates? (Compare Chapter 8, where the two sides run on different model families.)